In [1]:
import os
import itertools
import multiprocessing as mp

import numpy as np
import pandas as pd
import supervision as sv

from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict

from trackers import ByteTrackTracker
from trackers.eval import evaluate_mot_sequences


# Root directory for data and GT (this notebook is in the dancetrack folder).
DATA_ROOT = "."

DANCETRACK_DET_ROOT = os.path.join(DATA_ROOT, "dancetrack_yolox_dets")
DANCETRACK_GT_ROOT = os.path.join(DATA_ROOT, "TrackEval", "data", "gt", "dancetrack")


def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame.get(frame_id, []):
        det = line.split(",")  # "frame_no,x1,y1,x2,y2,det_conf"
        x1 = float(det[1])
        y1 = float(det[2])
        x2 = float(det[3])
        y2 = float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets

In [ ]:
def run_bytetrack_dancetrack_with_params(
    lost_track_buffer: int,
    track_activation_threshold: float,
    minimum_consecutive_frames: int,
    minimum_iou_threshold: float,
    high_conf_det_threshold: float,
    eval_set: str = "val",
    use_defaults: bool = False,
):
    """Run ByteTrack on all DanceTrack sequences for a given hyperparameter set."""

    assert eval_set in {"train", "val"}, f"Unsupported eval_set: {eval_set}"

    if use_defaults:
        tracker = ByteTrackTracker()
    else:
        tracker = ByteTrackTracker(
            lost_track_buffer=lost_track_buffer,
            track_activation_threshold=track_activation_threshold,
            minimum_consecutive_frames=minimum_consecutive_frames,
            minimum_iou_threshold=minimum_iou_threshold,
            high_conf_det_threshold=high_conf_det_threshold,
        )

    outputs_root = "BYTETRACK_outputs_dancetrack"
    os.makedirs(outputs_root, exist_ok=True)
    if use_defaults:
        save_dir = os.path.join(outputs_root, f"{eval_set}_defaults")
    else:
        save_dir = os.path.join(
            outputs_root,
            f"{eval_set}_L{lost_track_buffer}_"
            f"A{track_activation_threshold}_C{minimum_consecutive_frames}_I{minimum_iou_threshold}_H{high_conf_det_threshold}",
        )
    os.makedirs(save_dir, exist_ok=True)

    # Choose detections and GT depending on split
    det_root = os.path.join(DANCETRACK_DET_ROOT, eval_set)
    gt_dir = os.path.join(DANCETRACK_GT_ROOT, eval_set)
    seqmap = os.path.join(
        DANCETRACK_GT_ROOT, f"DanceTrack-{eval_set}.txt"
    )

    for seq in sorted(os.listdir(det_root)):
        if not seq.endswith(".txt"):
            continue

        tracker.reset()
        seq_name = seq.split(".")[0]

        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)

        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []

        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(
                    xyxy=raw_dets[:, :4],
                    confidence=raw_dets[:, 4],
                )
            else:
                dets = sv.Detections.empty()

            dets = tracker.update(detections=dets)

            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue

                width = right - left
                height = bottom - top

                output_lines.append(
                    f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                )

        save_path = os.path.join(save_dir, seq_name + ".txt")
        with open(save_path, "w") as f:
            f.writelines(output_lines)

    result = evaluate_mot_sequences(
        gt_dir=gt_dir,
        tracker_dir=save_dir,
        metrics=["CLEAR", "HOTA", "Identity"],
        seqmap=seqmap,
    )

    MOTA = result.to_dict()["aggregate"]["CLEAR"]["MOTA"]
    HOTA = result.to_dict()["aggregate"]["HOTA"]["HOTA"]
    IDF1 = result.to_dict()["aggregate"]["Identity"]["IDF1"]

    if use_defaults:
        print(f"ByteTrack | split:{eval_set} | defaults -> HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f}")
    else:
        print(
            f"ByteTrack | split:{eval_set} | L:{lost_track_buffer}, A:{track_activation_threshold}, "
            f"C:{minimum_consecutive_frames}, I:{minimum_iou_threshold}, H:{high_conf_det_threshold} -> "
            f"HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f}"
        )

    return {
        "lost_track_buffer": lost_track_buffer,
        "track_activation_threshold": track_activation_threshold,
        "minimum_consecutive_frames": minimum_consecutive_frames,
        "minimum_iou_threshold": minimum_iou_threshold,
        "high_conf_det_threshold": high_conf_det_threshold,
        "HOTA": HOTA,
        "IDF1": IDF1,
        "MOTA": MOTA,
    }

In [10]:
# Run ByteTrack with default parameters (ByteTrackTracker()) on the definitive eval set (val)
run_bytetrack_dancetrack_with_params(0, 0, 0, 0, 0, eval_set="val", use_defaults=True)

ByteTrack | split:val | defaults -> HOTA: 0.502, IDF1: 0.499, MOTA: 0.862


{'lost_track_buffer': 0,
 'track_activation_threshold': 0,
 'minimum_consecutive_frames': 0,
 'minimum_iou_threshold': 0,
 'high_conf_det_threshold': 0,
 'HOTA': 0.5023291374545561,
 'IDF1': 0.4993401492462432,
 'MOTA': 0.8620729475722636}

In [3]:
# Define ByteTrack hyperparameter search space

lost_track_buffer_candidates = [10, 30, 60, 90]
track_activation_threshold_candidates = [0.2, 0.5, 0.7, 0.9]
minimum_consecutive_frames_candidates = [1, 3]
minimum_iou_threshold_candidates = [0.05, 0.1, 0.3]
high_conf_det_threshold_candidates = [0.5, 0.6, 0.7, 0.9]

all_candidate_lists = [
    lost_track_buffer_candidates,
    track_activation_threshold_candidates,
    minimum_consecutive_frames_candidates,
    minimum_iou_threshold_candidates,
    high_conf_det_threshold_candidates,
]

parameter_combinations = list(itertools.product(*all_candidate_lists))
print(f"Total ByteTrack parameter combinations for DanceTrack: {len(parameter_combinations)}")

Total ByteTrack parameter combinations for DanceTrack: 384


In [4]:
def run_bytetrack_dancetrack_tuning_parallel(parameter_combinations, eval_set: str = "train"):
    num_workers = os.cpu_count()
    print(f"Running {len(parameter_combinations)} ByteTrack combinations on {eval_set} with {num_workers} workers")

    ctx = mp.get_context("fork")
    all_results = []

    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        futures = []
        for i, comb in enumerate(parameter_combinations):
            print(f"Submitting combination {i + 1}/{len(parameter_combinations)}")
            (
                lost_track_buffer,
                track_activation_threshold,
                minimum_consecutive_frames,
                minimum_iou_threshold,
                high_conf_det_threshold,
            ) = comb
            futures.append(
                executor.submit(
                    run_bytetrack_dancetrack_with_params,
                    lost_track_buffer,
                    track_activation_threshold,
                    minimum_consecutive_frames,
                    minimum_iou_threshold,
                    high_conf_det_threshold,
                    eval_set,
                )
            )

        for i, f in enumerate(futures):
            try:
                result = f.result()
                all_results.append(result)
                print(f"[{i + 1}/{len(parameter_combinations)}] combination finished.")
            except Exception as e:
                print(f"[{i + 1}/{len(parameter_combinations)}] combination FAILED: {e}")

    df = pd.DataFrame(all_results)
    print("ByteTrack hyperparameter tuning complete. Results stored in 'bytetrack_dancetrack_tuning_df'.")
    print(df.head())

    df.to_csv("bytetrack_dancetrack_tuning.csv", index=False)
    return df


bytetrack_dancetrack_tuning_df = run_bytetrack_dancetrack_tuning_parallel(parameter_combinations, eval_set="train")

best_idx = bytetrack_dancetrack_tuning_df["HOTA"].idxmax()
best_row = bytetrack_dancetrack_tuning_df.loc[best_idx]
print("\nBest ByteTrack HOTA row (train):\n", best_row)

best_params = dict(
    lost_track_buffer=int(best_row["lost_track_buffer"]),
    track_activation_threshold=float(best_row["track_activation_threshold"]),
    minimum_consecutive_frames=int(best_row["minimum_consecutive_frames"]),
    minimum_iou_threshold=float(best_row["minimum_iou_threshold"]),
    high_conf_det_threshold=float(best_row["high_conf_det_threshold"]),
)
print("Best params:", best_params)

Running 384 ByteTrack combinations on train with 10 workers
Submitting combination 1/384
Submitting combination 2/384
Submitting combination 3/384
Submitting combination 4/384
Submitting combination 5/384
Submitting combination 6/384
Submitting combination 7/384
Submitting combination 8/384
Submitting combination 9/384
Submitting combination 10/384
Submitting combination 11/384
Submitting combination 12/384
Submitting combination 13/384
Submitting combination 14/384
Submitting combination 15/384
Submitting combination 16/384
Submitting combination 17/384
Submitting combination 18/384
Submitting combination 19/384
Submitting combination 20/384
Submitting combination 21/384
Submitting combination 22/384
Submitting combination 23/384
Submitting combination 24/384
Submitting combination 25/384
Submitting combination 26/384
Submitting combination 27/384
Submitting combination 28/384
Submitting combination 29/384
Submitting combination 30/384
Submitting combination 31/384
Submitting combinat

In [5]:
# Final validation run using best_params from train tuning

if "best_params" not in globals() or best_params is None:
    raise RuntimeError("best_params is not defined. Run the tuning cell first.")

print("Running ByteTrack on DanceTrack val with:", best_params)

val_result = run_bytetrack_dancetrack_with_params(
    lost_track_buffer=best_params["lost_track_buffer"],
    track_activation_threshold=best_params["track_activation_threshold"],
    minimum_consecutive_frames=best_params["minimum_consecutive_frames"],
    minimum_iou_threshold=best_params["minimum_iou_threshold"],
    high_conf_det_threshold=best_params["high_conf_det_threshold"],
    eval_set="val",
)

Running ByteTrack on DanceTrack val with: {'lost_track_buffer': 60, 'track_activation_threshold': 0.9, 'minimum_consecutive_frames': 1, 'minimum_iou_threshold': 0.1, 'high_conf_det_threshold': 0.5}
ByteTrack | split:val | L:60, A:0.9, C:1, I:0.1, H:0.5 -> HOTA: 0.532, IDF1: 0.546, MOTA: 0.868
